# C1DB1B: Data Management for Decision-Making
## Workshop 3: Decision Modelling, MCDA & Sensitivity
### Executive decision: Where should Vårdlyftet establish its next primary care centre?

---

You still advise **Vårdlyftet**. You still compare **five municipalities**. You may still recommend that it should **not** proceed.

Module 2 prepared the evidence. Module 3 asks what that evidence suggests under **explicit preferences** and two checkable methods.

> Performance + Preferences + Method → Scores and Rankings

A high ranking does not automatically justify establishment.

---

### Sequence for this module

1. Check the scoring matrix.
2. Review overlap (correlation is a flag, not a deletion rule).
3. Calculate WSM, then TOPSIS, from the **same raw values**.
4. Compare rankings.
5. Test three weight profiles.
6. Prediction: keep **Not assigned**, or insert a justified predicted column from Module 2 (finite for all five).
7. Write a provisional recommendation with conditions.

Complete the **common synthetic benchmark** first. Then apply the same functions to **your** portfolio table from Module 2: the five rows and the approved scored columns you prepared, including group-collected indicators. Do not rebuild $X$ from the issued companions alone if your architecture has further columns.

---

### Data vintage
Paths and the predictive flag come from `data/manifest.json`. Do not hard-code file names.


---
### Who does what

**You, before the workshop:** bring a **laptop**. Run WSM and TOPSIS on the synthetic benchmark first, then on **your** Module 2 decision table (your KPIs, not only the demonstration columns). Arrive with draft rankings.

**The teacher, in the lecture:** works the A--E benchmark by hand and states the course conventions.

**The teacher, in the workshop:** checks the benchmark, releases one weight change, and compares your numbers with reference results.

**You, in the workshop:** freeze the baseline, predict the effect, rerun both methods on a **copy**, and explain agreement or disagreement.

A high rank is not yet a site choice. A solution notebook will be released after the workshop for the demonstration portfolio only.


---
## Part 1: Load the contract and the common benchmark

The benchmark alternatives A--E are **synthetic**. They are not municipalities.

Opportunity is a **Benefit** criterion (higher preferred). Annual cost is a **Cost** criterion (lower preferred). Baseline weights are 0.60 and 0.40.


In [ ]:
# ==============================================================================
# Load vintage, benchmark matrix, and metadata
# ==============================================================================
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 16)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

manifest_path = Path("data/manifest.json")
if not manifest_path.exists():
    raise FileNotFoundError("Run this notebook from the folder that contains data/.")

MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
m1 = MANIFEST["module1"]
m2 = MANIFEST["module2"]
m3 = MANIFEST["module3"]
T0 = int(m1["t0"])
T1 = int(m1["t1"])
PREDICTIVE_STATUS = m3["predictive_component"]
DOCUMENTED_MISSING = list(m2["documented_missing_markers"])

print(f"Course vintage       : {MANIFEST['vintage']}")
print(f"Predictive component : {PREDICTIVE_STATUS}")
print(f"Benchmark file       : {m3['benchmark_file']}")

bench = pd.read_csv(m3["benchmark_file"]).set_index("Alternative")
meta = pd.read_csv(m3["benchmark_meta_file"]).set_index("indicator")
meta = meta.reindex(bench.columns)

weights_b = meta["weight"].astype(float)
benefit_b = meta["direction"].eq("Benefit")

print("\nBenchmark matrix:")
print(bench.to_string())
print("\nMetadata:")
print(meta.to_string())
print("Weight sum:", float(weights_b.sum()))


---
## Shared scoring functions

These functions implement the course conventions:

- WSM: direction-aware min--max. A constant column becomes 0 for every alternative; keep its weight.
- TOPSIS: vector-normalize the **raw** matrix, then apply weights. Do not feed the WSM $Z$ matrix into TOPSIS.
- Ranks: higher score is better; ties use average rank.


In [ ]:
# ==============================================================================
# Course WSM and TOPSIS conventions
# ==============================================================================
def check_ready(X, weights, benefit):
    weights = weights.reindex(X.columns)
    benefit = benefit.reindex(X.columns)
    if not X.index.is_unique:
        raise ValueError("Row identifiers are not unique.")
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError("Scoring matrix contains missing or infinite values.")
    if weights.isna().any() or benefit.isna().any():
        raise ValueError("Every indicator needs a weight and a direction.")
    if (weights < 0).any():
        raise ValueError("Weights must be non-negative.")
    if not np.isclose(float(weights.sum()), 1.0):
        raise ValueError("Weights must sum to 1.")
    return weights, benefit.astype(bool)


def wsm_scores(X, weights, benefit):
    weights, benefit = check_ready(X, weights, benefit)
    span = X.max() - X.min()
    Z = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    constant = []
    for col in X.columns:
        if np.isclose(span[col], 0.0):
            Z[col] = 0.0
            constant.append(col)
        elif benefit[col]:
            Z[col] = (X[col] - X[col].min()) / span[col]
        else:
            Z[col] = (X[col].max() - X[col]) / span[col]
    score = Z.mul(weights, axis="columns").sum(axis=1)
    return score, Z, constant


def topsis_scores(X, weights, benefit):
    weights, benefit = check_ready(X, weights, benefit)
    col_norm = np.sqrt(X.pow(2).sum(axis=0))
    R = X.div(col_norm.where(col_norm.ne(0), 1.0), axis="columns")
    V = R.mul(weights, axis="columns")
    ideal = pd.Series(
        {c: V[c].max() if benefit[c] else V[c].min() for c in X.columns}
    )
    anti = pd.Series(
        {c: V[c].min() if benefit[c] else V[c].max() for c in X.columns}
    )
    d_plus = np.sqrt(V.sub(ideal, axis="columns").pow(2).sum(axis=1))
    d_minus = np.sqrt(V.sub(anti, axis="columns").pow(2).sum(axis=1))
    denom = d_plus + d_minus
    closeness = d_minus.div(denom.where(denom.ne(0)))
    return closeness, R, V, ideal, anti, d_plus, d_minus


def rank_higher_better(score):
    return score.rank(ascending=False, method="average")


def redistribute(weights, indicator, alpha):
    w = weights.astype(float).copy()
    if indicator not in w.index:
        raise KeyError(indicator)
    rest = 1.0 - float(w[indicator])
    if rest <= 0:
        raise ValueError("Cannot redistribute when the selected weight is 1.")
    w[indicator] = float(alpha)
    for k in w.index:
        if k != indicator:
            w[k] = float(weights[k]) * (1.0 - float(alpha)) / rest
    return w


In [ ]:
# ==============================================================================
# Decision exhibits (after the numerical checks). For the report, not extra methods.
# ==============================================================================
import matplotlib.pyplot as plt


def plot_wsm_contributions(Z, weights, title):
    contrib = Z.mul(weights, axis="columns")
    ax = contrib.plot(kind="bar", stacked=True, figsize=(8, 4.2), width=0.75)
    ax.set_ylabel("Contribution to WSM score")
    ax.set_xlabel("")
    ax.set_title(title)
    ax.legend(title="Criterion", bbox_to_anchor=(1.02, 1), loc="upper left")
    top = float(contrib.sum(axis=1).max())
    ax.set_ylim(0, max(1.05, top * 1.12))
    plt.tight_layout()
    plt.show()


def plot_rank_comparison(cmp, title):
    t = cmp.copy().sort_values("WSM_rank")
    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    ypos = list(range(len(t)))
    ax.plot(t["WSM_rank"], ypos, "o", markersize=10, label="WSM")
    ax.plot(t["TOPSIS_rank"], ypos, "s", markersize=9, label="TOPSIS")
    for i, (a, b) in enumerate(zip(t["WSM_rank"], t["TOPSIS_rank"])):
        ax.plot([a, b], [i, i], color="0.75", zorder=0)
    ax.set_yticks(ypos)
    ax.set_yticklabels(t.index.astype(str))
    ax.set_xlabel("Rank (1 = best)")
    ax.set_xlim(0.5, 5.5)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def plot_wsm_profiles(score_dict, title):
    ax = pd.DataFrame(score_dict).plot(kind="bar", figsize=(8, 4.2))
    ax.set_ylabel("WSM score")
    ax.set_xlabel("")
    ax.set_title(title)
    ax.legend(title="Weight profile")
    plt.tight_layout()
    plt.show()


---
## Task 1: Confirm modelling readiness

For the benchmark, confirm unique alternatives, finite values, directions aligned by name, and weights that sum to one.

Then build your **portfolio scoring matrix** from **your** Module 2 decision table. Include every approved numerical indicator that is finite for all five municipalities, including columns you joined from `data/group_sources/`. Issued companions are only the columns they already supplied.

In this vintage, **do not score** a travel-time field that is missing in source. Keep identifiers and status flags out of $X$.

**Self-check:** five unique codes; every scored column has a direction and a weight.


In [ ]:
# ==============================================================================
# TASK 1: Benchmark checks, then a five-municipality scoring matrix
# ==============================================================================
check_ready(bench, weights_b, benefit_b)
print("Benchmark readiness: PASS")

# TODO: replace with YOUR allocated codes
portfolio_codes = [1481, 1490, 1463, 1465, 1443]


def standardize_code(s):
    return s.astype("string").str.strip()


def parse_numeric(series, markers):
    raw = series.astype("string").str.strip()
    cleaned = raw.mask(raw.isin(list(markers)))
    numeric = pd.to_numeric(cleaned, errors="coerce")
    unexpected = cleaned.notna() & numeric.isna()
    if unexpected.any():
        raise ValueError("Unexpected tokens: " + str(cleaned.loc[unexpected].unique()))
    return numeric


anchor = pd.read_csv(m1["anchor_file"])
annual = pd.read_csv(m2["annual_file"], dtype={"Avg_Travel_Time_min": "string"})
static = pd.read_csv(m2["static_file"], dtype={"Est_Premises_Cost_kSEK": "string"})
for frame in (anchor, annual, static):
    frame["Municipality_Code"] = standardize_code(frame["Municipality_Code"])

codes = {str(c) for c in portfolio_codes}
demo = anchor[anchor["Municipality_Code"].isin(codes) & anchor["Year"].eq(T1)].copy()
hist = anchor[anchor["Municipality_Code"].isin(codes) & anchor["Year"].eq(T0)].copy()

annual["Avg_Travel_Time_min_num"] = parse_numeric(annual["Avg_Travel_Time_min"], DOCUMENTED_MISSING)
static["Est_Premises_Cost_kSEK_num"] = parse_numeric(static["Est_Premises_Cost_kSEK"], DOCUMENTED_MISSING)

merged = demo.merge(
    annual[["Municipality_Code", "Year", "Operating_Clinics", "Clinics_per_10k", "Avg_Travel_Time_min_num"]],
    on=["Municipality_Code", "Year"],
    how="left",
    validate="one_to_one",
)
merged = merged.merge(
    static[["Municipality_Code", "Care_Need_Index", "Est_Premises_Cost_kSEK_num"]],
    on="Municipality_Code",
    how="left",
    validate="one_to_one",
)
merged = merged.merge(
    hist[["Municipality_Code", "Pop_80_Plus"]].rename(columns={"Pop_80_Plus": "Pop_80_Plus_t0"}),
    on="Municipality_Code",
    how="left",
    validate="one_to_one",
)

merged["Share_80_plus_pct"] = 100.0 * merged["Pop_80_Plus"] / merged["Pop_Total"]
merged["Dependency_ratio"] = 100.0 * (
    merged["Pop_0_17"] + merged["Pop_65_79"] + merged["Pop_80_Plus"]
) / merged["Pop_18_64"]
merged["Growth_80_plus_pct"] = 100.0 * (
    merged["Pop_80_Plus"] - merged["Pop_80_Plus_t0"]
) / merged["Pop_80_Plus_t0"]

decision = merged.set_index("Municipality_Code")
decision.index.name = "Municipality_Code"

# Travel time is missing for Svenljunga in this vintage: exclude it from scoring.
scored = [
    "Share_80_plus_pct",
    "Dependency_ratio",
    "Growth_80_plus_pct",
    "Clinics_per_10k",
    "Care_Need_Index",
    "Est_Premises_Cost_kSEK_num",
]
X_port = decision[scored].astype(float)
X_port.columns = scored

benefit_p = pd.Series(
    {
        "Share_80_plus_pct": True,
        "Dependency_ratio": True,
        "Growth_80_plus_pct": True,
        "Clinics_per_10k": False,
        "Care_Need_Index": True,
        "Est_Premises_Cost_kSEK_num": False,
    }
)
weights_p = pd.Series(
    {
        "Share_80_plus_pct": 0.20,
        "Dependency_ratio": 0.15,
        "Growth_80_plus_pct": 0.20,
        "Clinics_per_10k": 0.15,
        "Care_Need_Index": 0.15,
        "Est_Premises_Cost_kSEK_num": 0.15,
    }
)

check_ready(X_port, weights_p, benefit_p)
print("\nPortfolio scoring matrix (issued companions — method demo):")
print(X_port.round(3).to_string())
print("Travel time excluded (source missing for at least one alternative).")
print("Teaching-value fields (CNI, clinics, premises cost) remain labelled as such.")
print("Portfolio readiness: PASS")

# --- Your architecture ---
# Module 2 writes data/group_sources/decision_table.csv. If you joined extra KPIs,
# load that table and replace `scored`, `benefit_p`, and `weights_p` with YOUR set.
# Uncomment and edit:
#
# own = pd.read_csv(
#     Path("data/group_sources/decision_table.csv"),
#     dtype={"Municipality_Code": "string"},
# ).set_index("Municipality_Code")
# scored = ["Share_80_plus_pct", "YOUR_KPI_2", "YOUR_KPI_3"]  # finite for all five
# X_port = own[scored].astype(float)
# benefit_p = pd.Series({"Share_80_plus_pct": True, "YOUR_KPI_2": True, "YOUR_KPI_3": False})
# weights_p = pd.Series({"Share_80_plus_pct": 0.4, "YOUR_KPI_2": 0.3, "YOUR_KPI_3": 0.3})
# check_ready(X_port, weights_p, benefit_p)
# print("\nPortfolio scoring matrix (YOUR architecture):")
# print(X_port.round(3).to_string())


---
### Your analysis for Task 1

**Which columns did you exclude from scoring, and why?**
*(Write here.)*

**Are clinic counts / CNI / premises costs labelled teaching values in your metadata?**
*(Write here.)*


---
## Task 2: Review overlap and correlation

Inspect definitions first. Then compute Pearson correlations on the portfolio matrix.

Treat $|r|>0.8$ as a **review flag**, not an automatic deletion. With five alternatives, high or low correlation is weak evidence on its own.

**Self-check:** no indicator is removed solely because it crossed 0.8.


In [ ]:
# ==============================================================================
# TASK 2: Definition review and correlation flags
# ==============================================================================
corr = X_port.corr(method="pearson")
print("Pearson correlation (n = 5 alternatives):")
print(corr.round(3).to_string())

flag_pairs = []
cols = list(X_port.columns)
for i, a in enumerate(cols):
    for b in cols[i + 1 :]:
        r = corr.loc[a, b]
        if abs(r) > 0.8:
            flag_pairs.append((a, b, float(r)))
print("\nFlagged pairs |r| > 0.8:")
print(flag_pairs if flag_pairs else "none")
print("Action in this demonstration: retain flagged pairs with a logged overlap note.")


---
### Your analysis for Task 2

**Definition overlap (before looking at $r$):**
*(Write here.)*

**Flagged pairs and your decision (retain / request change / reweight):**
*(Write here.)*


---
## Task 3: Weighted Sum Model

Use direction-aware min--max, then $S_i=\sum w_j z_{ij}$.

Benchmark reference scores: A 0.60, B 0.65, C 0.60, D 0.50, E 0.40.

Then apply the same function to the portfolio.


In [ ]:
# ==============================================================================
# TASK 3: WSM on the benchmark, then the portfolio
# ==============================================================================
wsm_b, Z_b, const_b = wsm_scores(bench, weights_b, benefit_b)
print("Benchmark Z:")
print(Z_b.round(3).to_string())
print("Constant columns:", const_b)
print("WSM scores:", wsm_b.round(4).to_dict())
assert np.allclose(wsm_b.loc[["A", "B", "C", "D", "E"]], [0.60, 0.65, 0.60, 0.50, 0.40])
print("Benchmark WSM check: PASS")

wsm_p, Z_p, const_p = wsm_scores(X_port, weights_p, benefit_p)
print("\nPortfolio WSM scores:")
print(wsm_p.sort_values(ascending=False).round(4).to_string())
print("Constant columns:", const_p)


---
### Your analysis for Task 3

**What does a WSM score of 0.65 mean for benchmark B? What does it *not* mean?**
*(Write here.)*


---
### Exhibit: WSM contributions

Each bar is the weighted stack $w_j z_{ij}$. Use it in the report to show *why* a score is high, not as an extra method. The benchmark picture matches Lecture 3.


In [ ]:
plot_wsm_contributions(Z_b, weights_b, "Benchmark: WSM contributions (baseline weights)")
plot_wsm_contributions(Z_p, weights_p, "Portfolio: WSM contributions (baseline weights)")


---
## Task 4: TOPSIS

Start from the **raw** matrix. Vector-normalize, weight, build ideal/anti-ideal profiles, then closeness $C_i=D_i^-/(D_i^++D_i^-)$.

Benchmark closeness (3 d.p.): A 0.565, B 0.648, C 0.588, D 0.487, E 0.435.


In [ ]:
# ==============================================================================
# TASK 4: TOPSIS on the benchmark, then the portfolio
# ==============================================================================
c_b, R_b, V_b, ideal_b, anti_b, dp_b, dm_b = topsis_scores(bench, weights_b, benefit_b)
topsis_ref = pd.Series({"A": 0.5650, "B": 0.6475, "C": 0.5877, "D": 0.4872, "E": 0.4350})
print("Benchmark closeness:")
print(c_b.round(4).to_string())
print("D+:", dp_b.round(4).to_dict())
print("D-:", dm_b.round(4).to_dict())
assert np.allclose(c_b.reindex(topsis_ref.index), topsis_ref, atol=5e-4)
print("Benchmark TOPSIS check: PASS")
print("Ideal is not required to be an actual alternative:", dict(ideal_b.round(4)))

c_p, R_p, V_p, ideal_p, anti_p, dp_p, dm_p = topsis_scores(X_port, weights_p, benefit_p)
print("\nPortfolio TOPSIS closeness:")
print(c_p.sort_values(ascending=False).round(4).to_string())


---
### Your analysis for Task 4

**Why must TOPSIS not start from the WSM min-max matrix?**
*(Write here.)*


---
## Task 5: Compare rankings

One table: municipality, WSM score/rank, TOPSIS score/rank. Align by identifier before comparing. Spearman $\rho$ is descriptive only.

Do not compare the *size* of a WSM score with a TOPSIS closeness.


In [ ]:
# ==============================================================================
# TASK 5: Side-by-side ranks and Spearman rho
# ==============================================================================
def compare_methods(wsm, topsis):
    out = pd.DataFrame({"WSM_score": wsm, "TOPSIS_score": topsis})
    out["WSM_rank"] = rank_higher_better(out["WSM_score"])
    out["TOPSIS_rank"] = rank_higher_better(out["TOPSIS_score"])
    if out[["WSM_rank", "TOPSIS_rank"]].nunique().min() > 1:
        rho = out["WSM_rank"].corr(out["TOPSIS_rank"], method="spearman")
    else:
        rho = np.nan
    return out, rho

cmp_b, rho_b = compare_methods(wsm_b, c_b)
cmp_p, rho_p = compare_methods(wsm_p, c_p)
print("Benchmark comparison:")
print(cmp_b.round(4).to_string())
print("Spearman rho (benchmark):", rho_b)
print("\nPortfolio comparison:")
print(cmp_p.sort_values("WSM_rank").round(4).to_string())
print("Spearman rho (portfolio):", rho_p)


---
### Your analysis for Task 5

**Same leader? Who changes place? What in the profiles could explain it?**
*(Write here.)*


---
### Exhibit: WSM vs TOPSIS ranks

Rank 1 is best. Markers that do not overlap are a method disagreement. On the demonstration portfolio this should show Svenljunga vs Bollebygd.


In [ ]:
plot_rank_comparison(cmp_b, "Benchmark ranks (1 = best)")
plot_rank_comparison(cmp_p, "Portfolio ranks (1 = best)")


---
## Task 6: Three weight profiles

1. Baseline (already used).
2. Equal weights.
3. A priority profile: raise one criterion and **redistribute** the rest so weights still sum to one.

Keep data and method conventions unchanged. Run **both** methods each time.


In [ ]:
# ==============================================================================
# TASK 6: Baseline, equal, and cost-priority profiles
# ==============================================================================
def profile_leader(X, w, benefit, name):
    wsm, _, _ = wsm_scores(X, w, benefit)
    tops, *_ = topsis_scores(X, w, benefit)
    tab, rho = compare_methods(wsm, tops)
    wsm_lead = tab["WSM_score"].idxmax()
    tops_lead = tab["TOPSIS_score"].idxmax()
    print(f"\n=== {name} ===")
    print("weights:", {k: round(float(v), 4) for k, v in w.items()})
    print("WSM leader:", wsm_lead, "TOPSIS leader:", tops_lead, "rho:", rho)
    return tab

_ = profile_leader(bench, weights_b, benefit_b, "Benchmark baseline 0.60/0.40")
w_eq_b = pd.Series(1.0 / len(weights_b), index=weights_b.index)
_ = profile_leader(bench, w_eq_b, benefit_b, "Benchmark equal")
w_cost = redistribute(weights_b, "annual_cost", 0.70)
_ = profile_leader(bench, w_cost, benefit_b, "Benchmark cost-priority 0.70")

print("\n----- Portfolio -----")
_ = profile_leader(X_port, weights_p, benefit_p, "Portfolio baseline")
w_eq_p = pd.Series(1.0 / len(weights_p), index=weights_p.index)
_ = profile_leader(X_port, w_eq_p, benefit_p, "Portfolio equal")
w_pri = redistribute(weights_p, "Est_Premises_Cost_kSEK_num", 0.40)
_ = profile_leader(X_port, w_pri, benefit_p, "Portfolio premises-cost priority 0.40")


---
### Your analysis for Task 6

**Did the leader change? Under which profile, and what does that mean for Vårdlyftet?**
*(Write here.)*


---
### Exhibit: WSM scores under the three weight profiles

Same alternatives and data; only preferences change. Report the tested profiles rather than writing that the result is robust.


In [ ]:
wsm_eq_b, _, _ = wsm_scores(bench, w_eq_b, benefit_b)
wsm_cost_b, _, _ = wsm_scores(bench, w_cost, benefit_b)
plot_wsm_profiles(
    {
        "baseline 0.60/0.40": wsm_b,
        "equal": wsm_eq_b,
        "cost 0.70": wsm_cost_b,
    },
    "Benchmark WSM scores under three profiles",
)
wsm_eq_p, _, _ = wsm_scores(X_port, w_eq_p, benefit_p)
wsm_pri_p, _, _ = wsm_scores(X_port, w_pri, benefit_p)
plot_wsm_profiles(
    {
        "baseline": wsm_p,
        "equal": wsm_eq_p,
        "premises-cost 0.40": wsm_pri_p,
    },
    "Portfolio WSM scores under three profiles",
)


---
## Task 7: Predictive modelling, if the group took it up

The issued common path is **Not assigned**. If the group justified a model or an external forecast in Module 2, a predicted column may enter the matrix only if it is finite for all five municipalities and labelled as a prediction. Do not invent a target merely to fill the matrix.


In [ ]:
# ==============================================================================
# TASK 7: Predictive component status
# ==============================================================================
print("Issued predictive flag (common path):", PREDICTIVE_STATUS)
print("If the group did not take up prediction: leave the MCDA matrix without a predicted column.")
print("If the group did: a predicted column may enter only if justified in Module 2 and finite for all five.")


---
### Your analysis for Task 7

**Status:** *(Write here.)*


---
## Task 8: Provisional recommendation

State a preferred municipality, a conditional preference, or why no recommendation is yet justified. Name the main trade-off, which tested weights change the conclusion, and which gaps remain.

You are not required to force WSM and TOPSIS to agree.


---
### Your analysis for Task 8

**Provisional advice to Vårdlyftet:**
*(Write here.)*


---
## Live weight-change rehearsal (benchmark)

Preserve the baseline. Predict, then set annual cost to 0.70 using the redistribution formula, rerun both methods, and explain.

In class the instructor may issue a different $\alpha$ on this same benchmark. Do not invent a new indicator during the challenge.


In [ ]:
# ==============================================================================
# Live-challenge rehearsal: cost weight -> 0.70 on the benchmark copy
# ==============================================================================
print("Baseline WSM:", wsm_b.round(4).to_dict())
print("Prediction: cheaper alternatives should improve; E need not automatically win TOPSIS.")
w_live = redistribute(weights_b, "annual_cost", 0.70)
assert np.isclose(w_live.sum(), 1.0)
wsm_live, _, _ = wsm_scores(bench, w_live, benefit_b)
c_live, *_ = topsis_scores(bench, w_live, benefit_b)
cmp_live, rho_live = compare_methods(wsm_live, c_live)
print("Updated weights:", w_live.round(4).to_dict())
print(cmp_live.round(4))
print("WSM leader:", wsm_live.idxmax(), "TOPSIS leader:", c_live.idxmax())
print("Reference WSM under 0.30/0.70: E = 0.70 (lecture).")
assert np.isclose(wsm_live.loc["E"], 0.70)
print("Challenge rehearsal check: PASS")


---
### Your analysis for the challenge rehearsal

**What changed, and what did not? Why is this not yet an establishment decision?**
*(Write here.)*


---
## Gateway 3 closing check

Can the group explain which alternative leads under the baseline, why the methods agree or disagree, what could change the result, and why the highest rank may still need conditions?

Bring: checked benchmark, portfolio ranks, three profiles, and unresolved questions. The three-page report stays outside this notebook.
